# 01_state_graph

对应 `index.md` 第 1 阶段：StateGraph 骨架 —— State、节点、静态边、`compile` / `invoke`。

笔记本在 `codes/langgraph/01_state_graph/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT` 并加载 `codes/langgraph/local.env`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

`StateGraph(State)` 只是画布，能 `invoke` 的是 `graph.compile()` 的返回值。本阶段先把骨架搭熟：State 是什么、节点返回什么、边怎么连、reducer 怎么合并。第 1–3 题不接模型，第 4 题起真的调 API。

In [1]:
import os
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "01_state_graph" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langgraph" / "01_state_graph"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

def load_local_env(env_path: Path) -> None:
    """把 local.env 写入 os.environ（已有键不覆盖）。"""
    if not env_path.is_file():
        return
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("\"'")
        if key and key not in os.environ:
            os.environ[key] = value

ROOT = lab_root()
load_local_env(ROOT.parent / "local.env")
ROOT, bool(os.environ.get("DEEPSEEK_API_KEY"))

(PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langgraph/01_state_graph'),
 True)

## 1. 最小图：一个节点

已有 State `Counter`（一个 `count` 字段）和节点函数 `increment`（把 `count` 加一后返回）。

把它接成一张图并跑一次：

1. `graph = StateGraph(Counter)`
2. `graph.add_node("increment", increment)`
3. 用静态边连起来：`START → increment → END`
4. `app = graph.compile()`
5. `app.invoke({"count": 0})`，打印返回的 state

`START` / `END` 是引入的哨兵，用来标出入口和出口。

In [2]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

class Counter(TypedDict):
    count: int

def increment(state: Counter) -> dict:
    return {"count": state["count"] + 1}

# 作答

from langgraph.graph import START, END, StateGraph

graph = StateGraph(Counter)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)
app = graph.compile()

print(app.invoke({"count": 0}))
#评阅
# 对。五步齐全：StateGraph(Counter) → add_node → START → increment → END → compile → invoke，输出 {'count': 1}。
# 一处冗余：前置代码已 import 过 START/END/StateGraph，作答里又 import 了一遍；能跑，但没必要。

#参考答案
# graph = StateGraph(Counter)
# graph.add_node("increment", increment)
# graph.add_edge(START, "increment")
# graph.add_edge("increment", END)
# app = graph.compile()
# print(app.invoke({"count": 0}))

{'count': 1}


## 2. 默认 reducer 是覆盖，`add_messages` 才是追加

节点返回的是**部分 state**：`{"字段": 新值}`。字段怎么和旧值合并，由 **reducer** 决定。默认行为是**覆盖**（新值直接换掉旧值）；消息这类要累加的列表必须显式给 reducer。

已有节点 `add_reply`（返回一条 `AIMessage("新消息")`）。定义两个 State：

- `OverwriteState`：`items: list`
- `AppendState`：`items: Annotated[list, add_messages]`

分别把 `add_reply` 接成 `START → add_reply → END` 的图，`compile` 后各 `invoke({"items": [AIMessage("旧消息")]})`。

打印两份结果的 `items` **长度**和内容，观察：覆盖只剩一条，追加有两条。

In [8]:
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

def add_reply(state) -> dict:
    return {"items": [AIMessage("新消息")]}

# 作答

class OverwriteState(TypedDict):
    items: list

class AppendState(TypedDict):
    items: Annotated[list, add_messages]

graph = StateGraph(OverwriteState)

graph.add_node("add_reply", add_reply)
graph.add_edge(START, "add_reply")
graph.add_edge("add_reply", END)

app = graph.compile()

print(app.invoke({"items": [AIMessage("旧消息")]}))

print("---" * 20)

graph = StateGraph(AppendState)

graph.add_node("add_reply", add_reply)
graph.add_edge(START, "add_reply")
graph.add_edge("add_reply", END)

app = graph.compile()

print(app.invoke({"items": [AIMessage("旧消息")]}))
#评阅
# 对，修好后两张图都对：OverwriteState 只剩 1 条「新消息」，AppendState 有 2 条（旧消息、新消息）。
# 你在这题踩过两次坑，正好是重点：State 字段名、节点返回的键、invoke 传入的键必须逐字一致；对不上时 LangGraph 不报错，只静默丢掉更新——轻则 None，重则整份 state 为空。
# 一处没做到：题面要「打印 items 的长度和内容」，你只打了整个 state，内容能看出，但没有显式的 len。

#参考答案
# class OverwriteState(TypedDict):
#     items: list
#
# class AppendState(TypedDict):
#     items: Annotated[list, add_messages]
#
# for State, label in [(OverwriteState, "覆盖"), (AppendState, "追加")]:
#     g = StateGraph(State)
#     g.add_node("add_reply", add_reply)
#     g.add_edge(START, "add_reply")
#     g.add_edge("add_reply", END)
#     out = g.compile().invoke({"items": [AIMessage("旧消息")]})["items"]
#     print(label, len(out), [m.content for m in out])

{'items': [AIMessage(content='新消息', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}
------------------------------------------------------------
{'items': [AIMessage(content='旧消息', additional_kwargs={}, response_metadata={}, id='c28f6532-cc67-467f-b566-27f8f2ae8aa7', tool_calls=[], invalid_tool_calls=[]), AIMessage(content='新消息', additional_kwargs={}, response_metadata={}, id='892b52d5-9f4d-4772-9990-6e42da64bc89', tool_calls=[], invalid_tool_calls=[])]}


## 3. 用预置的 `MessagesState` 累积两轮

字段名固定叫 `messages`、reducer 已经配好 `add_messages`，LangGraph 自带 `MessagesState` 替你把上面那层 `Annotated` 省掉。

已有节点 `ack`（返回一条 `AIMessage("收到")`）。用 `MessagesState` 建图（`START → ack → END`），`compile` 后：

1. 先 `invoke({"messages": [HumanMessage("第一问")]})`，记下结果
2. 把结果的全部 `messages` 再加上一条 `HumanMessage("第二问")`，作为第二次 `invoke` 的输入
3. 打印第二次最终 `messages` 的**条数**，以及每条的类型名（`type(m).__name__`）

第二轮应当能看到「人类 → AI → 人类 → AI」四条，说明历史被累积而不是覆盖。

In [20]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, MessagesState, StateGraph

def ack(state: MessagesState) -> dict:
    return {"messages": [AIMessage("收到")]}

# 作答

graph = StateGraph(MessagesState)

graph.add_node("ack", ack)
graph.add_edge(START, "ack")
graph.add_edge("ack", END)

app = graph.compile()

response = app.invoke({"messages": [HumanMessage("第一问")]})

response = app.invoke({"messages": response["messages"] + [HumanMessage("第二问")]})

print(len(response["messages"]))

for m in response["messages"]:
    print(type(m).__name__, m.content)
#评阅
# 对。第一次 invoke 拿到「第一问 / 收到」两条；第二次把 response["messages"] 拼上「第二问」再 invoke，最终 4 条，顺序正是「人类 → AI → 人类 → AI」。
# 之前 message 少个 s 的问题已修好——字段名三处对齐，和上一题是同一个坑。
# 小点：变量复用了 response，第一次的结果被覆盖。这里没问题（右侧先取值再赋值），但留个 r1 更清楚。

#参考答案
# response = app.invoke({"messages": [HumanMessage("第一问")]})
# response = app.invoke({"messages": response["messages"] + [HumanMessage("第二问")]})
# print(len(response["messages"]))
# for m in response["messages"]:
#     print(type(m).__name__, m.content)

4
HumanMessage 第一问
AIMessage 收到
HumanMessage 第二问
AIMessage 收到


## 4. 真实模型节点

图里的节点就是普通函数，把 `model.invoke(...)` 放进去，就是最常见的一种节点。

已有 `model`（`init_chat_model`，deepseek）和 `MessagesState`。

写节点 `agent(state)`：调用 `model.invoke(state["messages"])`，把回复包成 `{"messages": [回复]}` 返回。用 `MessagesState` 建图（`START → agent → END`），`compile` 后 `invoke({"messages": [HumanMessage("用一句话介绍 LangGraph。")]})`，打印**最后一条**消息的 `content`。

这一题开始会真的等 API，比前面慢。

In [21]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, MessagesState, StateGraph

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)

# 作答

def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}
graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_edge(START, "agent")
graph.add_edge("agent", END)

app = graph.compile()

print(app.invoke({"messages": [HumanMessage("用一句话介绍 LangGraph。")]})["messages"][-1].content)
#评阅
# 对。节点就是普通函数：model.invoke(state["messages"])，把回复包成 {"messages": [回复]} 返回，正是 partial update 的写法；最后 [-1].content 取最后一条也对。
# 小点：函数定义和 graph = StateGraph(...) 之间没留空行，读起来挤；结构本身没问题。

#参考答案
# def agent(state: MessagesState) -> dict:
#     return {"messages": [model.invoke(state["messages"])]}
#
# graph = StateGraph(MessagesState)
# graph.add_node("agent", agent)
# graph.add_edge(START, "agent")
# graph.add_edge("agent", END)
# out = graph.compile().invoke({"messages": [HumanMessage("用一句话介绍 LangGraph。")]})
# print(out["messages"][-1].content)

LangGraph 是一个用图结构编排有状态、可循环、多智能体协作的 LLM 应用框架。


## 5. 两个节点串成流水线

静态边可以把多个节点串起来：`START → draft → polish → END`。每个节点只改自己关心的字段，更新会依次合并进同一份 state。

已有 State `Doc`（`text: str`）、节点 `draft`（在前面加 `草稿：`）和 `polish`（在结尾加 `！`）。注意 `text` 用的是默认 reducer（覆盖），所以后一个节点拿到的是前一个节点写好的值。

把 `draft`、`polish` 依次接进图，`compile` 后 `invoke({"text": "LangGraph"})`，打印最终的 `text`。

In [22]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

class Doc(TypedDict):
    text: str

def draft(state: Doc) -> dict:
    return {"text": "草稿：" + state["text"]}

def polish(state: Doc) -> dict:
    return {"text": state["text"] + "！"}

# 作答

graph = StateGraph(Doc)

graph.add_node("draft", draft)
graph.add_node("polish", polish)

graph.add_edge(START, "draft")
graph.add_edge("draft", "polish")
graph.add_edge("polish", END)

app = graph.compile()

print(app.invoke({"text": "LangGraph"}))
#评阅
# 对。draft、polish 依次接成 START → draft → polish → END；text 用默认覆盖 reducer，后一个节点拿到前一个写好的值，最终「草稿：LangGraph！」。

#参考答案
# graph = StateGraph(Doc)
# graph.add_node("draft", draft)
# graph.add_node("polish", polish)
# graph.add_edge(START, "draft")
# graph.add_edge("draft", "polish")
# graph.add_edge("polish", END)
# print(graph.compile().invoke({"text": "LangGraph"})["text"])

{'text': '草稿：LangGraph！'}
